# Validate: `_intra_drawdown`

Checks that `_intra_drawdown(returns)` correctly computes the worst peak-to-trough
decline within a single trade cycle.

```python
def _intra_drawdown(returns: pd.Series) -> float:
    # Prepend 1.0 so the entry price is the initial peak — without this,
    # a trade that drops on bar 1 would show zero drawdown.
    eq = pd.concat([pd.Series([1.0]), (1 + returns).cumprod()])
    return float((eq / eq.cummax() - 1).min())

```

The function takes the **bar-level returns within a single trade** and returns
the largest peak-to-trough drop expressed as a fraction (always ≤ 0).

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hailmary.analytics.signal_analytics import _intra_drawdown
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Hand-Calculated Cases

Each case has an analytically known answer so we can assert correctness directly.

| Case | Returns | Equity curve | Expected DD |
|---|---|---|---|
| All gains | `[+10%, +10%, +10%]` | Always at new highs | `0.0` |
| Flat | `[0%, 0%, 0%]` | Constant | `0.0` |
| Single loss | `[-20%]` | Drops straight to 0.80 | `-0.20` |
| Up then down | `[+10%, -20%]` | Peak 1.10 → trough 0.88 | `-0.20` |
| Down then up | `[-20%, +10%]` | Peak 1.0 → trough 0.80 | `-0.20` |
| Two separate dips | `[+20%, -15%, +20%, -15%]` | Each dip from its own peak | `−0.15` |
| Accumulating losses | `[-10%, -10%, -10%]` | 0.90 → 0.81 → 0.729 | `≈ -0.271` |

In [ ]:
cases = [
    ("All gains",          [0.10, 0.10, 0.10],            0.0),
    ("Flat",               [0.0,  0.0,  0.0],             0.0),
    ("Single loss",        [-0.20],                       -0.20),
    ("Up then down",       [0.10, -0.20],                 -0.20),
    ("Down then up",       [-0.20, 0.10],                 -0.20),
    ("Two separate dips",  [0.20, -0.15, 0.20, -0.15],   -0.15),
    ("Accumulating losses",[-0.10, -0.10, -0.10],         0.9**3 - 1),  # -0.271
]

rows = []
all_pass = True
for name, rets, expected in cases:
    result = _intra_drawdown(pd.Series(rets))
    ok = abs(result - expected) < 1e-9
    all_pass = all_pass and ok
    rows.append({
        "Case":     name,
        "Returns":  str([f"{r:+.0%}" for r in rets]),
        "Expected": f"{expected:.4f}",
        "Got":      f"{result:.4f}",
        "Pass":     "✓" if ok else "✗ FAIL",
    })

results_df = pd.DataFrame(rows).set_index("Case")
display(results_df)
print()
print("All cases pass" if all_pass else "*** FAILURES DETECTED ***")

## 2. Visual Walkthrough

Plot equity curve, running peak, and drawdown for a few illustrative cases.
Each panel shows exactly what the function is computing.

In [ ]:
def plot_case(rets_list: list[float], title: str) -> go.Figure:
    s = pd.Series(rets_list)
    eq = (1 + s).cumprod()
    peak = eq.cummax()
    dd = eq / peak - 1
    x = list(range(len(eq)))

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.06,
    )

    # Running peak
    fig.add_trace(go.Scatter(
        x=x, y=peak,
        name="Running peak",
        line=dict(color=PALETTE["text_secondary"], width=1, dash="dot"),
    ), row=1, col=1)

    # Equity curve
    fig.add_trace(go.Scatter(
        x=x, y=eq,
        name="Equity",
        line=dict(color=PALETTE["accent_blue"], width=2),
        mode="lines+markers",
        marker=dict(size=7),
    ), row=1, col=1)

    # Drawdown
    fig.add_trace(go.Scatter(
        x=x, y=dd,
        name="Intra-trade DD",
        fill="tozeroy",
        line=dict(color=PALETTE["accent_red"], width=1.5),
        fillcolor="rgba(248,81,73,0.15)",
    ), row=2, col=1)

    worst = float(dd.min())
    fig.add_hline(
        y=worst, row=2, col=1,
        line=dict(color=PALETTE["accent_orange"], width=1, dash="dash"),
        annotation_text=f"  worst: {worst:.2%}",
        annotation_font_color=PALETTE["accent_orange"],
    )

    apply_theme(fig, title=title, height=380)
    fig.update_layout(
        yaxis=dict(title_text="Equity"),
        yaxis2=dict(title_text="Drawdown", tickformat=".0%"),
    )
    return fig


plot_case([0.10, -0.20], "Up then down: +10%, −20%").show()
plot_case([0.20, -0.15, 0.20, -0.15], "Two dips: +20%, −15%, +20%, −15%").show()
plot_case([-0.10, -0.10, -0.10], "Accumulating losses: −10%, −10%, −10%").show()

## 2b. Dollar Walkthrough

Same maths shown in dollar terms with a $10,000 starting stake.

In [ ]:
stake = 10_000

examples = {
    "Up then down (+10%, -20%)":          [0.10, -0.20],
    "Down then up (-20%, +10%)":          [-0.20, 0.10],
    "Two dips (+20%, -15%, +20%, -15%)":  [0.20, -0.15, 0.20, -0.15],
}

for title, rets in examples.items():
    s = pd.Series(rets)
    # Prepend 1.0 as in the fixed function
    eq_norm = pd.concat([pd.Series([1.0]), (1 + s).cumprod()]).reset_index(drop=True)
    eq_usd  = eq_norm * stake
    peak_usd = eq_usd.cummax()
    dd_usd   = eq_usd - peak_usd      # absolute $ drop from peak
    dd_pct   = eq_usd / peak_usd - 1  # same as the function output

    rows = []
    for i, (nav, pk, drop, pct) in enumerate(zip(eq_usd, peak_usd, dd_usd, dd_pct)):
        label = "entry" if i == 0 else f"bar {i}"
        rows.append({
            "Bar":        label,
            "Return":     f"{rets[i-1]:+.0%}" if i > 0 else "—",
            "NAV ($)":    f"${nav:,.0f}",
            "Peak ($)":   f"${pk:,.0f}",
            "Drop ($)":   f"−${abs(drop):,.0f}" if drop < 0 else "—",
            "DD (%)":     f"{pct:.1%}" if pct < 0 else "—",
        })

    df_show = pd.DataFrame(rows).set_index("Bar")
    worst_dd  = dd_pct.min()
    worst_usd = dd_usd.min()
    print(f"{'─'*55}")
    print(f"  {title}")
    print(f"  _intra_drawdown = {worst_dd:.1%}   (${abs(worst_usd):,.0f} drop from peak)")
    display(df_show)
    print()

## 3. Edge Cases

Boundary conditions that could silently produce wrong results.

In [ ]:
edge_cases = [
    # Single bar in trade (init == exit bar) — no intra-trade movement possible
    ("Single bar, gain",   [0.05],   0.0),
    ("Single bar, loss",   [-0.05],  -0.05),
    # Return of exactly zero — equity flat, no drawdown
    ("Single zero return", [0.0],    0.0),
    # Big winner with a small mid-trade dip.
    # eq = [1.30, 1.274, 1.6562], peak = [1.30, 1.30, 1.6562]
    # dd[1] = 1.274 / 1.30 - 1 = -0.02  (the dip return itself)
    ("Big win, tiny dip",  [0.30, -0.02, 0.30],  -0.02),
]

edge_rows = []
all_edge_pass = True
for name, rets, expected in edge_cases:
    result = _intra_drawdown(pd.Series(rets))
    ok = abs(result - expected) < 1e-9
    all_edge_pass = all_edge_pass and ok
    edge_rows.append({
        "Case":     name,
        "Returns":  str([f"{r:+.2%}" for r in rets]),
        "Expected": f"{expected:.6f}",
        "Got":      f"{result:.6f}",
        "Pass":     "✓" if ok else "✗ FAIL",
    })

display(pd.DataFrame(edge_rows).set_index("Case"))
print()
print("All edge cases pass" if all_edge_pass else "*** FAILURES DETECTED ***")

## 4. Sanity-Check Against Real Backtest Trades

Pull actual bar-level returns for a known trade from `bt_result` and verify that
`_intra_drawdown` applied to those bars matches the `max_intra_drawdown_net` value
that `trade_stats()` stored.

In [ ]:
import pandas as pd
from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics.signal_analytics import SignalTradePerformance

signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

bars      = yahoo.get_bars(symbols, start=start - pd.offsets.BDay(signal.warmup), end=end, adjust=False)
signal_df = signal.run(bars, trim_start=start)
bt_result = BarBacktest().run(signal_df)
trade_perf = SignalTradePerformance(bt_result)

trades = trade_perf.trade_stats()
df     = bt_result.data

mismatches = []
for _, row in trades.iterrows():
    sym = row["symbol"]
    trade_bars = df.loc[sym].loc[row["entry_date"]:row["exit_date"], "return_net"]
    computed_dd = _intra_drawdown(trade_bars)
    stored_dd   = row["max_intra_drawdown_net"]
    if abs(computed_dd - stored_dd) > 1e-9:
        mismatches.append({
            "symbol":     sym,
            "entry_date": row["entry_date"],
            "computed":   computed_dd,
            "stored":     stored_dd,
        })

print(f"Checked {len(trades)} trades across {trades['symbol'].nunique()} symbols")
if mismatches:
    print(f"*** {len(mismatches)} MISMATCHES ***")
    display(pd.DataFrame(mismatches))
else:
    print("All stored drawdowns match recomputed values — function is consistent")

## 5. Visualise a Real Trade

Pick the trade with the largest intra-trade drawdown and plot the bar-level equity
and drawdown curve within that trade to see what the function actually measured.

In [ ]:
worst_trade = trades.loc[trades["max_intra_drawdown_net"].idxmin()]
sym = worst_trade["symbol"]
entry, exit_ = worst_trade["entry_date"], worst_trade["exit_date"]

trade_rets = df.loc[sym].loc[entry:exit_, "return_net"].reset_index(drop=True)

print(f"Worst trade: {sym}  {entry.date()} → {exit_.date()}")
print(f"  Net return:       {worst_trade['return_net']:+.2%}")
print(f"  Intra-trade DD:   {worst_trade['max_intra_drawdown_net']:+.2%}")
print(f"  Duration (bars):  {worst_trade['duration']}")

plot_case(trade_rets.tolist(), f"Real trade: {sym}  {entry.date()} → {exit_.date()}").show()